In [1]:
%pip install --user optuna
%pip install --user plotly
import plotly
import numpy as np
import os
import optuna
from optuna.trial import TrialState
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.transforms import v2
import copy
from sklearn.metrics import roc_auc_score


os.chdir('/users/pbernius/melanoma_cnn')

!nvidia-smi
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Thu Nov 12 10:27:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro RTX 6000                Off |   00000000:86:00.0 Off |                    0 |
| N/A   19C    P8             12W /  250W |       3MiB /  23040MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

#!kill 59 62 92 210
!ps -eo pid,user,etime,cmd | grep -i ipykernel
!nvidia-smi

     63 pbernius       00:16 /usr/bin/python3 -m ipykernel_launcher -f /tmpscratch/pbernius/.local/share/jupyter/runtime/kernel-af6d617d-698f-4a43-8a99-b78e68906768.json
    104 pbernius       00:00 /bin/bash -c ps -eo pid,user,etime,cmd | grep -i ipykernel
    106 pbernius       00:00 grep -i ipykernel
Thu Nov 12 10:27:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro RTX 6

In [3]:

data = np.load("melanoma_training_data.npy", allow_pickle=True)
cutoff = int(0.8 * len(data))

training_data = data[:cutoff]
val_data = data[cutoff:]



# stack images into one tensor (N, H, W, 3), reorder to PyTorch's (N, 3, H, W), then copy into contiguous memory
train_x = torch.Tensor(np.array([item[0] for item in training_data])).permute(0, 3, 1, 2).contiguous()
#normalize data
#train_x = train_x/255


#one-hot vector labels tensor
train_y = torch.Tensor([item[1] for item in training_data])



# stack images into one tensor (N, H, W, 3), reorder to PyTorch's (N, 3, H, W), then copy into contiguous memory
val_x = torch.Tensor(np.array([item[0] for item in val_data])).permute(0, 3, 1, 2).contiguous()
#normalize data
#val_x = val_x/255


#one-hot vector labels tensor
val_y = torch.Tensor([item[1] for item in val_data])





/tmp/ipykernel_63/4153494329.py:16: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  train_y = torch.Tensor([item[1] for item in training_data])


In [4]:
from IPython.core.completerlib import module_list
import torch
import torch.nn as nn
import torch.nn.functional as F


#define image size 50x50 pixels
img_size = 100



class TunableNet(nn.Module):

    def __init__(self, channels, fc_units, num_pooling, adaptivepool, dropout2d=0.05, dropout=0.1, kernel_size=3, pooling2d=2):

        # call constructor method from parent class
        super().__init__()

        self.convolutional_layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()
        self.pooling_layers = nn.ModuleList()
        self.dropout_layers = nn.ModuleList()
        

        in_channels = 3

        for i, out_channels in enumerate(channels):

            self.convolutional_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size, padding=kernel_size // 2))
            self.norm_layers.append(nn.BatchNorm2d(out_channels))
            self.pooling_layers.append(nn.MaxPool2d(pooling2d) if i < num_pooling else nn.Identity())
            self.dropout_layers.append(nn.Dropout2d(dropout2d))

            in_channels = out_channels


        
        
        self.adaptive_pool = nn.AdaptiveAvgPool2d(adaptivepool)
        self.flatten = nn.Flatten(1)
        self.fc1 = nn.Linear(in_channels * adaptivepool * adaptivepool, fc_units)
        self.fc_dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(fc_units, 2)




    def forward(self, x):

        for conv, norm, pool, drop in zip(self.convolutional_layers, self.norm_layers, self.pooling_layers, self.dropout_layers):
            x = conv(x)
            x = norm(x)
            x = F.relu(x)
            x = pool(x)
            x = drop(x)

        x = self.adaptive_pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc_dropout(x)
        x = self.fc2(x)

        

        


        return x



In [5]:
!chmod u+w /users/pbernius/melanoma_cnn

In [6]:
import os, getpass, tempfile
print(getpass.getuser(), os.getuid(), os.getgid())
print(os.environ.get("HOME"), tempfile.gettempdir())
for p in ["/users/pbernius", "/users/pbernius/melanoma_cnn", os.environ.get("HOME"), "/tmp"]:
    print(p, os.path.exists(p), os.access(p, os.W_OK))

pbernius 243439 243439
/tmpscratch/pbernius /tmp
/users/pbernius True True
/users/pbernius/melanoma_cnn True True
/tmpscratch/pbernius True True
/tmp True True


In [7]:
!id
!ls -ld /users/pbernius /users/pbernius/melanoma_cnn
!mount | grep -i users

uid=243439(pbernius) gid=243439(pbernius) groups=243439(pbernius),65534(nogroup)
drwxr-x--- 16 pbernius pbernius 4096 Nov  3 08:29 /users/pbernius
drwx------  5 pbernius pbernius 4096 Nov 12 09:59 /users/pbernius/melanoma_cnn
tmpfs on /users/pbernius type tmpfs (rw,nosuid,nodev,relatime,size=65536k,uid=243439,gid=243439,inode64)
superior-data.crc.nd.edu:/primary_users on /users type nfs4 (rw,nosuid,nodev,relatime,vers=4.2,rsize=65536,wsize=65536,namlen=255,hard,proto=tcp,nconnect=16,timeo=600,retrans=2,sec=sys,clientaddr=10.32.89.21,local_lock=none,addr=10.32.123.60)


In [ ]:
from PIL.ImageEnhance import Brightness
from alembic.util import exc
from torch.nn.functional import adaptive_max_pool1d_with_indices




#number of epochs
epochs = 50










def objective(trial):

    torch.manual_seed(42)
    np.random.seed(42)


    # ---------------- training hyperparameters ----------------
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
 
    learning_rate = trial.suggest_float("learning_rate", 0.00003, 0.0005, log=True)
 
    # Adam (L2 penalty) and AdamW (decoupled decay) need very different weight decay ranges,
    # so each optimizer gets its own conditional parameter
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW"])
    if optimizer_name == "Adam":
        weight_decay = trial.suggest_float("weight_decay_adam", 0.00001, 0.001, log=True)
    else:
        weight_decay = trial.suggest_float("weight_decay_adamw", 0.001, 0.5, log=True)
 
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.1)
 


    # set loss function (label smoothing only during training)
    train_loss_function = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    val_loss_function = nn.CrossEntropyLoss()
 
 
    # ---------------- architecture ----------------
    n_layers = trial.suggest_int("n_layers", 2, 8)
 
    base_channels = trial.suggest_categorical("base_channels", [32, 48, 64, 80, 96])
    channel_growth = trial.suggest_categorical("channel_growth", [2, 3, 4])
    channels = [base_channels * 2 ** min(i, channel_growth) for i in range(n_layers)]
 
    kernel_size = trial.suggest_categorical("kernel_size", [3, 4, 5, 6, 7])
 
    num_pooling = trial.suggest_int("num_pooling", 2, min(5, n_layers))
 

    adaptive_pool = trial.suggest_categorical("adaptivepool", [4, 6, 8, 10, 12])
    adp_pool = min(adaptive_pool, img_size//2**num_pooling)
    trial.set_user_attr("adaptivepool_used", adp_pool)
 
 


    # ---------------- regularization ----------------
    params = {
        "fc_units": trial.suggest_categorical("fc_units", [64, 128, 256, 512]),
        "num_pooling": num_pooling,
        "adaptivepool": adp_pool,
        "kernel_size": kernel_size,
        "dropout2d": trial.suggest_float("dropout2d", 0.0, 0.5),
        "dropout": trial.suggest_float("dropout", 0.1, 0.6)
    }
 
 
    # ---------------- data augmentation ----------------
    horizontal_flip_p = trial.suggest_float("horizontal_flip_p", 0.0, 0.5)
    vertical_flip_p = trial.suggest_float("vertical_flip_p", 0.0, 0.5)
    rotation = trial.suggest_categorical("rotation", [0, 30, 90, 180])
    brightness = trial.suggest_float("brightness", 0.0, 0.25)
    contrast = trial.suggest_float("contrast", 0.0, 0.25)
    saturation = trial.suggest_float("saturation", 0.0, 0.15)
    hue = trial.suggest_float("hue", 0.0, 0.02)
    crop = trial.suggest_float("crop", 0.6, 1.0)
 
 
    training_augmentation = v2.Compose([
        v2.RandomHorizontalFlip(p=horizontal_flip_p),
        v2.RandomVerticalFlip(p=vertical_flip_p),
        v2.ColorJitter(brightness=brightness, contrast=contrast, saturation=saturation, hue=hue),
        v2.RandomRotation(rotation),
        v2.RandomResizedCrop(size=img_size, scale=(crop, 1.0))
    ])
 
 
 
 
    model = TunableNet(channels, **params, )
 
 
    # ---------------- learning rate schedule ----------------
    schedule_factor = trial.suggest_float("schedule_factor", 0.3, 0.5)
    sched_patience = trial.suggest_int("sched_patience", 3, 10)
 
 
    # set optimizer
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
 
    lr_sched = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=schedule_factor, patience=sched_patience)



    best_auc = 0.0
    best_val_loss = np.inf
    patience = 12

    epochs_without_improvement = 0

    try:

        num_params = sum(p.numel() for p in model.parameters())
        if num_params > 25000000:
            print(f"Number of parameters is {num_params}. Pruning trial")
            raise optuna.TrialPruned()
        else:
            print(f"Training model with {num_params} parameters.")


        model = model.to(DEVICE)


        #dummy pass so optuna doesn't crash on bad combinations
        model.eval()
        with torch.no_grad():
            try:
                x = torch.zeros(1, 3, img_size, img_size).to(DEVICE)
                for conv, norm, pool in zip(model.convolutional_layers, model.norm_layers, model.pooling_layers):
                    x = pool(F.relu(norm(conv(x))))
                print(f"trial {trial.number}: feature map before adaptive pool = {tuple(x.shape)}")
            except RuntimeError as e:
                print(e)
                raise optuna.TrialPruned()

        for epoch in range(epochs):

            



            #data shuffler
            perm = torch.randperm(len(train_x))
            
            model.train()
        
            
        
            
            for i in range(0, len(training_data), batch_size):
        
                #index for shuffling data
                idx = perm[i: i+batch_size]
        
        
                #print(f"EPOCH {epoch+1}, fraction complete: {i/len(train_x)}")
        
                batch_x = train_x[idx].float() / 255
                batch_y = train_y[idx]
        
                #for colab gpu
                batch_x = batch_x.to(DEVICE)
                batch_y = batch_y.to(DEVICE)
        
                #resetting model parameters to zero each pass
                optimizer.zero_grad()
        
                augmented_image = torch.stack([training_augmentation(img) for img in batch_x])
            
                #pass batch of images to model
                outputs = model(augmented_image)
        
                #compute errors
                loss = train_loss_function(outputs, batch_y)
                n = len(batch_x)
        
                
                
        
                #backpropagation to calculate gradient of of loss with respect to model parameters
                loss.backward()
        
        
                #update model parameters based on previously computed gradient
                optimizer.step()
        
        


            model.eval()


            val_loss = 0.0
            val_samples = 0
            all_scores = []
            all_labels = []
        
            with torch.no_grad():
        
                for j in range(0, len(val_data), batch_size):
        
                    val_batch_x = val_x[j: j+batch_size].float() / 255
                    val_batch_y = val_y[j: j+batch_size]
        
                    #for colab gpu
                    val_batch_x = val_batch_x.to(DEVICE)
                    val_batch_y = val_batch_y.to(DEVICE)
            
            
                    #pass batch of images to model
                    outputs = model(val_batch_x)
            
                    #compute errors
                    n = len(val_batch_x)
                    val_loss += val_loss_function(outputs, val_batch_y).item() * n
                    val_samples += n
        
                    
                    probs = torch.softmax(outputs, dim=1)[:, 1]
                    labels = val_batch_y[:, 1]
        
                    all_scores.append(probs)
                    all_labels.append(labels)
        
            val_loss = val_loss / val_samples
        
            lr_sched.step(val_loss)
        
            all_scores = torch.cat(all_scores).cpu().numpy()
            all_labels = torch.cat(all_labels).cpu().numpy()
        
            val_auc = roc_auc_score(all_labels, all_scores)


            score = val_auc
            best_auc = max(best_auc, score)
            trial.report(best_auc, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()


            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_without_improvement = 0
            else:
                epochs_without_improvement +=1
                if epochs_without_improvement >= patience:
                    print(f"Stopping early at Epoch: {epoch+1}")
                    break
        
        

        return best_auc

    finally:
        del model, optimizer, lr_sched; torch.cuda.empty_cache()







study = optuna.create_study(direction="maximize", 
                            sampler=optuna.samplers.TPESampler(seed=42), 
                            storage="sqlite:///study.db", 
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=25, n_warmup_steps=30),
                            study_name="melanoma_pass6", 
                            load_if_exists=True)




if len(study.trials) == 0:
    study.enqueue_trial({
        "label_smoothing": 0.0,
        "dropout2d": 0.05,
        "dropout": 0.1,
        "horizontal_flip_p": 0.5,
        "vertical_flip_p": 0.5,
        "rotation": 0,
        "brightness": 0.12,
        "contrast": 0.12,
        "saturation": 0.08,
        "hue": 0.008,
        "crop": 1.0,
        "schedule_factor": 0.5,
        "sched_patience": 3,
    })


study.optimize(objective, n_trials=200, catch=(RuntimeError, ValueError))

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

[I 2026-11-12 10:27:35,788] Using an existing study with `study_name='melanoma_pass6'` instead of creating a new one.


Training model with 2103362 parameters.
trial 116: feature map before adaptive pool = (1, 192, 12, 12)


[I 2026-11-12 10:34:46,028] Trial 116 pruned. 


Training model with 2103362 parameters.
trial 117: feature map before adaptive pool = (1, 192, 12, 12)


[I 2026-11-12 10:46:04,606] Trial 117 finished with value: 0.9847565580192374 and parameters: {'label_smoothing': 0.004317662242581135, 'dropout2d': 0.03575766605131266, 'dropout': 0.11781763070193457, 'horizontal_flip_p': 0.4885447573165731, 'vertical_flip_p': 0.4213169169036363, 'rotation': 0, 'brightness': 0.05860977544680422, 'contrast': 0.12368500458443031, 'saturation': 0.07379926576553175, 'hue': 0.008568531762446274, 'crop': 0.9738509231424332, 'schedule_factor': 0.4727765415736816, 'sched_patience': 3}. Best is trial 62 with value: 0.985800791021248.


Training model with 2103362 parameters.
trial 118: feature map before adaptive pool = (1, 192, 12, 12)


... (288 outputs left out)

[I 2026-11-13 08:31:50,246] Trial 263 pruned. 


Training model with 2103362 parameters.
trial 264: feature map before adaptive pool = (1, 192, 12, 12)


[W 2026-11-13 08:33:16,860] Trial 264 failed with parameters: {'label_smoothing': 0.0020508878906420496, 'dropout2d': 0.0055298265744026, 'dropout': 0.20282550158450074, 'horizontal_flip_p': 0.3842752544916483, 'vertical_flip_p': 0.427640562408894, 'rotation': 0, 'brightness': 0.06376919802659381, 'contrast': 0.11658890796571504, 'saturation': 0.06993313222513456, 'hue': 0.010101366765764551, 'crop': 0.9809907683641994, 'schedule_factor': 0.4893756171503609, 'sched_patience': 3} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/tmpscratch/pbernius/.local/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_63/3389240687.py", line 165, in objective
    augmented_image = torch.stack([training_augmentation(img) for img in batch_x])
  File "/tmp/ipykernel_63/3389240687.py", line 165, in <listcomp>
    augmented_image = torch.stack([training_augmentation(img) f

KeyboardInterrupt: 

In [9]:
study = optuna.load_study(study_name="melanoma_pass6", storage="sqlite:///study.db")

In [10]:
import optuna.visualization as vis

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()
vis.plot_parallel_coordinate(study).show()

In [11]:
df = study.trials_dataframe()
df.sort_values("value", ascending=False)


,number,value,datetime_start,datetime_complete,duration,params_brightness,params_contrast,params_crop,params_dropout,params_dropout2d,...,params_label_smoothing,params_rotation,params_saturation,params_sched_patience,params_schedule_factor,params_vertical_flip_p,user_attrs_adaptivepool_used,system_attrs_fixed_params,system_attrs_tpe:relative_params:0,state
238,238,0.985855,2026-11-13 04:46:23.930598,2026-11-13 04:58:17.188291,0 days 00:11:53.257693,0.071331,0.123987,0.992975,0.193926,0.005157,...,0.001557,0,0.079717,3,0.482151,0.434082,10,NaN,"{""brightness"": 0.07133057983486168, ""contrast""...",COMPLETE
62,62,0.985801,2026-11-12 01:25:20.591412,2026-11-12 01:37:43.271723,0 days 00:12:22.680311,0.088486,0.137706,0.977066,0.154459,0.011809,...,0.001590,0,0.068306,3,0.484444,0.436092,10,NaN,"{""brightness"": 0.08848618984603047, ""contrast""...",COMPLETE
163,163,0.985757,2026-11-12 17:41:34.995732,2026-11-12 17:53:13.784460,0 days 00:11:38.788728,0.066244,0.152807,0.990385,0.138888,0.017041,...,0.003246,0,0.065933,3,0.490086,0.391778,10,NaN,"{""brightness"": 0.06624439067175787, ""contrast""...",COMPLETE
49,49,0.985663,2026-11-11 23:06:30.239922,2026-11-11 23:18:39.318369,0 days 00:12:09.078447,0.054338,0.122228,0.960514,0.147997,0.040389,...,0.008638,0,0.047109,4,0.488927,0.496211,10,NaN,"{""brightness"": 0.054338072614133696, ""contrast...",COMPLETE
216,216,0.985658,2026-11-13 01:31:00.493732,2026-11-13 01:42:52.135842,0 days 00:11:51.642110,0.057341,0.138603,0.963272,0.202976,0.008016,...,0.009085,0,0.080241,3,0.488237,0.404421,10,NaN,"{""brightness"": 0.05734119877349917, ""contrast""...",COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2,2,0.967895,2026-11-11 13:39:45.111469,2026-11-11 13:54:19.174283,0 days 00:14:34.062814,0.196294,0.049918,0.618580,0.315973,0.209903,...,0.030424,180,0.077135,3,0.421509,0.305926,10,NaN,NaN,COMPLETE
7,7,0.967127,2026-11-11 14:47:46.241945,2026-11-11 15:00:31.966923,0 days 00:12:45.724978,0.062323,0.102596,0.630792,0.112710,0.171016,...,0.052273,180,0.113333,3,0.357950,0.015715,10,NaN,NaN,COMPLETE
1,1,0.958245,2026-11-11 13:25:27.342700,2026-11-11 13:39:43.783414,0 days 00:14:16.440714,0.177018,0.005146,0.684936,0.465997,0.380286,...,0.037454,90,0.145486,3,0.336365,0.078009,10,NaN,NaN,COMPLETE
115,115,NaN,2026-11-12 09:52:34.360466,2026-11-12 09:59:09.231034,0 days 00:06:34.870568,0.042029,0.117040,0.983528,0.189975,0.003646,...,0.001023,0,0.084608,3,0.489111,0.423041,10,NaN,"{""brightness"": 0.04202942840243181, ""contrast""...",FAIL
